In [1]:
import torch
import torch.nn as nn

device = torch.accelerator.current_accelerator () .type if torch.accelerator.is_available () else "cpu"
print ( f"Using {device} device" )
print ( torch.__version__ )

Using cuda device
2.11.0+cu130


Model , Loss , Optimizer
Typical pytorch pipeline looks like:
I. Design model ( input , output , forward pass with different layers )
II. Construct loss and optimizer
III. Training loop:
    - forward : compute prediction and loss
    - backward : compute gradients
    - update weights

In [3]:
X = torch.tensor ( [ [ 1 ] , [ 2 ] , [ 3 ] , [ 4 ] , [ 5 ] , [ 6 ] , [ 7 ] , [ 8 ] ] , dtype = torch.float32 ) ; print ( f"{X=} {X.grad=} {X.is_leaf=}" )
Y = torch.tensor ( [ [ 2 ] , [ 4 ] , [ 6 ] , [ 8 ] , [ 10 ] , [ 12 ] , [ 14 ] , [ 16 ] ] , dtype = torch.float32 ) ; print ( f"{Y=} {Y.grad=} {Y.is_leaf=}" )

n_samples , n_features = X.shape
print ( f"{n_samples=} {n_features=}" )

X_test = torch.tensor ( [ [ 5 ] ] , dtype = torch.float32 )


X=tensor([[1.],
        [2.],
        [3.],
        [4.],
        [5.],
        [6.],
        [7.],
        [8.]]) X.grad=None X.is_leaf=True
Y=tensor([[ 2.],
        [ 4.],
        [ 6.],
        [ 8.],
        [10.],
        [12.],
        [14.],
        [16.]]) Y.grad=None Y.is_leaf=True
n_samples=8 n_features=1


I. Design Model
Use a built-in model and implement the forward pass

In [4]:
class LinearRegression ( nn.Module ) :
    def __init__ ( self , input_dim , output_dim ) :
        super ( LinearRegression , self ) .__init__ () # call the constructor of the parent class nn.Module
        # definne layers
        self.linear = nn.Linear ( in_features = input_dim , out_features = output_dim ) # create a linear layer with input_dim input features and output_dim output features

    def forward ( self , x ) :
        return self.linear ( x ) # pass the input through the linear layer and return the output

input_size , output_size = n_features , n_features
model = LinearRegression ( input_size , output_size ) ; print ( f"{model=}" )
print ( f"f({X_test=}) = {model ( X_test ) .item () :.1f}" )

model=LinearRegression(
  (linear): Linear(in_features=1, out_features=1, bias=True)
)
f(X_test=tensor([[5.]])) = -2.4


II. Define loss and optimizer

In [9]:
learning_rate = 0.01
n_epochs = 100

loss = nn.MSELoss () # mean squared error loss function
optimizer = torch.optim.SGD ( model.parameters () , lr = learning_rate ) # stochastic gradient

for epoch in range ( n_epochs ) :
    #model.train () # set the model to training mode (not strictly necessary here since we don't have dropout or batchnorm layers, but it's good practice)
    y_predicted = model ( X ) # forward pass: compute the predicted y by passing x to the model
    l = loss ( Y , y_predicted ) # compute the loss
    l.backward () # backward pass: compute gradient of the loss with respect to model parameters
    optimizer.step () # update the parameters using the gradients computed in the backward pass
    optimizer.zero_grad () # zero the gradients before running the backward pass
    if ( epoch + 1 ) % 10 == 0 :
        w , b = model.parameters () # get the parameters of the model (weights and bias)
        print ( f"{epoch + 1=} : {l.item()=:.4f} {w[0][0].item()=}, {l.item()=} {b[0].item()=}" )
    optimizer.zero_grad () # zero the gradients before running the backward pass

    #loss.backward () # backward pass: compute gradient of the loss with respect to model parameters
    #optimizer.step () # update the parameters using the gradients computed in the backward pass
print ( f"f({X_test.item()=}) = {model ( X_test ) .item () :.3f}" )

epoch + 1=10 : l.item()=0.0006 w[0][0].item()=1.990358591079712, l.item()=0.0006099777529016137 b[0].item()=0.05420714616775513
epoch + 1=20 : l.item()=0.0006 w[0][0].item()=1.990736722946167, l.item()=0.0005630766390822828 b[0].item()=0.05208147317171097
epoch + 1=30 : l.item()=0.0005 w[0][0].item()=1.9910999536514282, l.item()=0.0005197768332436681 b[0].item()=0.050039127469062805
epoch + 1=40 : l.item()=0.0005 w[0][0].item()=1.991448998451233, l.item()=0.0004798163427039981 b[0].item()=0.04807690158486366
epoch + 1=50 : l.item()=0.0004 w[0][0].item()=1.9917842149734497, l.item()=0.00044291859376244247 b[0].item()=0.04619160667061806
epoch + 1=60 : l.item()=0.0004 w[0][0].item()=1.992106318473816, l.item()=0.00040885951602831483 b[0].item()=0.044380269944667816
epoch + 1=70 : l.item()=0.0004 w[0][0].item()=1.9924159049987793, l.item()=0.00037742831045761704 b[0].item()=0.04263998195528984
epoch + 1=80 : l.item()=0.0003 w[0][0].item()=1.9927133321762085, l.item()=0.0003484055923763662

III. Training loop